# Motion-Guided Self-Supervised Learning for Cardiac Cine MRI
## Notebook 05: Motion Estimation & Temporal Consistency Module

This notebook demonstrates the **Motion Estimation & Differentiable Warping Pipeline** designed to leverage cardiac cine temporal dynamics.

### Core Pipeline Components:
1. **`SimpleFlowNet`**: A lightweight, fully differentiable 2D CNN (117k parameters) that estimates dense 2D displacement fields $(u_x, u_y)$ between consecutive cine frames $[I_t, I_{t+1}]$.
2. **`SpatialTransformer`**: A differentiable spatial warping operator utilizing `F.grid_sample` to warp images, intermediate feature maps, and segmentation masks.
3. **Self-Supervised Motion Consistency**: Combined objective comprising photometric intensity matching $\mathcal{L}_{\text{photo}}$ and Total Variation smoothness regularization $\mathcal{L}_{\text{smooth}}$.

> **COMPUTE CONSTRAINT NOTE**:
> In strict accordance with project requirements, actual multi-epoch training is performed on a dedicated GPU training system (`TRAINING MACHINE ONLY`).
> This notebook runs a lightweight CPU demonstration without executing expensive training loops.

In [ ]:
import os
import sys
from pathlib import Path

# Ensure project root is in sys.path
project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import yaml
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

from src.motion import (
    SimpleFlowNet,
    SpatialTransformer,
    MotionEstimator,
    warp_features,
    warp_mask,
    compute_temporal_consistency_loss,
)
from src.dataset import ACDCTemporalDataset

print(f"Project root: {project_root}")
print(f"PyTorch version: {torch.__version__}")

### 1. Load Motion Configuration
We inspect the configuration in `configs/motion.yaml`.

In [ ]:
config_path = project_root / "configs" / "motion.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

print("Motion Configuration:")
print(f"  Method:              {config['motion_model']['method']}")
print(f"  Channels:            {config['motion_model']['channels']}")
print(f"  Warping mode:        {config['warping']['mode']}")
print(f"  Align corners:       {config['warping']['align_corners']}")
print(f"  Photometric weight:  {config['loss']['photometric_weight']}")
print(f"  Smoothness weight:   {config['loss']['smoothness_weight']}")

### 2. Load Adjacent Frame Pair $(I_t, I_{t+1})$
We load an adjacent frame pair from `ACDCTemporalDataset`. Frames are strictly from the same patient and sequential cine phases.

In [ ]:
data_cfg = config['data']
dataset = ACDCTemporalDataset(
    processed_dir=str(project_root / data_cfg['processed_dir']),
    split_file=str(project_root / data_cfg['train_split']),
)

sample = dataset[50]
frame_t = sample['frame_t'].unsqueeze(0)   # (1, 1, 256, 256)
frame_t1 = sample['frame_t1'].unsqueeze(0) # (1, 1, 256, 256)

print(f"Patient:       {sample['patient_id']}")
print(f"Slice Index:   {sample['slice_idx']}")
print(f"Frame indices: t={sample['frame_idx_t']} -> t+1={sample['frame_idx_t1']}")
print(f"Frame shape:   {frame_t.shape}")

### 3. Initialize MotionEstimator
We instantiate `MotionEstimator` on CPU and inspect its parameter count.

In [ ]:
device = torch.device('cpu')
motion_estimator = MotionEstimator(
    channels=config['motion_model']['channels'],
    align_corners=config['warping']['align_corners'],
    padding_mode=config['warping']['padding_mode'],
    photometric_weight=config['loss']['photometric_weight'],
    smoothness_weight=config['loss']['smoothness_weight'],
).to(device)

num_params = sum(p.numel() for p in motion_estimator.parameters() if p.requires_grad)
print(f"MotionEstimator initialized on {device}.")
print(f"Trainable parameters: {num_params:,} (lightweight CNN)")

### 4. Motion Estimation & Forward Warping
We estimate the displacement field and warp frame $t$ toward frame $t+1$.

In [ ]:
with torch.no_grad():
    output = motion_estimator(frame_t, frame_t1)

flow = output['flow']        # (1, 2, 256, 256)
warped_t = output['warped_t']# (1, 1, 256, 256)

print(f"Displacement field shape: {flow.shape} (dx, dy)")
print(f"Warped image shape:       {warped_t.shape}")
print(f"Photometric Loss:         {output['photo_loss'].item():.4f}")
print(f"Smoothness Loss:          {output['smooth_loss'].item():.4f}")
print(f"Total Motion Loss:        {output['total_loss'].item():.4f}")

### 5. Visualizing Displacement Field and Motion Compensation
We display the source frame, target frame, flow magnitude, warped frame, and residual difference.

In [ ]:
img_t = frame_t.squeeze().numpy()
img_t1 = frame_t1.squeeze().numpy()
img_warped = warped_t.squeeze().numpy()

dx = flow[0, 0].numpy()
dy = flow[0, 1].numpy()
flow_mag = np.sqrt(dx**2 + dy**2)

raw_err = np.abs(img_t1 - img_t)
warped_err = np.abs(img_t1 - img_warped)

fig, axes = plt.subplots(1, 5, figsize=(22, 4.5))

axes[0].imshow(img_t, cmap='gray')
axes[0].set_title("Source Frame (t)")
axes[0].axis('off')

axes[1].imshow(img_t1, cmap='gray')
axes[1].set_title("Target Frame (t+1)")
axes[1].axis('off')

im_flow = axes[2].imshow(flow_mag, cmap='viridis')
axes[2].set_title(f"Flow Magnitude\n(Max: {flow_mag.max():.2f} px)")
axes[2].axis('off')
plt.colorbar(im_flow, ax=axes[2], fraction=0.046, pad=0.04)

axes[3].imshow(img_warped, cmap='gray')
axes[3].set_title("Warped Frame: W(t, flow)")
axes[3].axis('off')

im_err = axes[4].imshow(warped_err, cmap='inferno')
axes[4].set_title(f"Residual Error |t+1 - W(t)|\n(Mean: {warped_err.mean():.4f})")
axes[4].axis('off')
plt.colorbar(im_err, ax=axes[4], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

### 6. Feature-Level Warping Demonstration
When enforcing feature-level temporal consistency between encoder stages, the spatial resolution of intermediate features is lower than the image. The module automatically resizes and scales the displacement field to match feature map dimensions.

In [ ]:
# Simulate feature map at encoder stage 1 (C=64, H=128, W=128)
dummy_feature = torch.randn(1, 64, 128, 128)
warped_feature = motion_estimator.warp_features(dummy_feature, flow)

print(f"Input feature shape:  {dummy_feature.shape}")
print(f"Warped feature shape: {warped_feature.shape}")
print("Feature warping successfully matched spatial dimensions.")

### 7. Mask Warping Demonstration
During semi-supervised fine-tuning, motion warping allows propagating labeled masks or high-confidence pseudo-labels from frame $t$ to frame $t+1$.

In [ ]:
dummy_mask = torch.zeros(1, 256, 256, dtype=torch.long)
# Create simulated circular myocardium / LV region
Y, X = torch.meshgrid(torch.arange(256), torch.arange(256), indexing='ij')
r = torch.sqrt((X - 128)**2 + (Y - 128)**2)
dummy_mask[r < 30] = 1   # LV
dummy_mask[(r >= 30) & (r < 45)] = 2  # Myocardium

warped_mask = motion_estimator.warp_mask(dummy_mask, flow, num_classes=4)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(dummy_mask[0].numpy(), cmap='tab10', vmin=0, vmax=3)
axes[0].set_title("Original Mask (t)")
axes[0].axis('off')

axes[1].imshow(warped_mask[0].numpy(), cmap='tab10', vmin=0, vmax=3)
axes[1].set_title("Warped Mask: W(Mask_t, flow)")
axes[1].axis('off')
plt.tight_layout()
plt.show()

### 8. Gradient Flow & Trainability Verification
We verify that the motion loss propagates finite gradients through `SimpleFlowNet` and `SpatialTransformer`.

In [ ]:
motion_estimator.train()
optimizer = torch.optim.AdamW(motion_estimator.parameters(), lr=1e-4)
optimizer.zero_grad()

loss_dict = motion_estimator(frame_t, frame_t1)
total_loss = loss_dict['total_loss']
total_loss.backward()

grads = [p.grad for p in motion_estimator.flownet.parameters() if p.requires_grad]
has_valid_grads = all(g is not None and not torch.isnan(g).any() for g in grads)

print(f"Total motion loss:  {total_loss.item():.4f}")
print(f"All layers received valid gradients: {has_valid_grads}")
optimizer.step()
print("Optimizer step completed successfully.")

### 9. Execution on Separate GPU Training System (`TRAINING MACHINE ONLY`)

To execute motion-guided pretraining and fine-tuning on the separate GPU training server:

```bash
# ========================================================
# TRAINING MACHINE ONLY (DO NOT RUN ON DEV SYSTEM)
# ========================================================
python src/motion.py --config configs/motion.yaml --device cuda
```

The motion estimator and spatial transformer integrate seamlessly into the self-supervised and semi-supervised fine-tuning stages.